# DNN Programming Assignment 1
**Binary Classification — Logistic Regression vs MLP from scratch**

Using the Breast Cancer Wisconsin dataset. Both models built with NumPy only (no sklearn models).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

np.random.seed(42)
%matplotlib inline

## 1. Dataset Selection (1 mark)

Going with Breast Cancer Wisconsin (Diagnostic) from UCI. It has 569 samples and 30 features which clears the minimum requirements. It's a binary classification problem — malignant vs benign tumors.

**Why Recall as primary metric?** In medical diagnosis, a false negative (telling someone they don't have cancer when they do) is way worse than a false positive. So we want to maximize recall.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print(f'Samples: {X.shape[0]}')
print(f'Features: {X.shape[1]}')
print(f'Classes: {list(data.target_names)}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
X.describe().T[['mean', 'std', 'min', 'max']]

Quick look at the data — features have very different scales (e.g. `mean area` goes up to 2501 while `mean smoothness` maxes at 0.16). Definitely need to standardize before training.

In [ ]:
# check for missing values
print('Missing values per feature:')
print(X.isnull().sum().sum())
# no missing values, good

In [ ]:
# class balance check
y.value_counts().plot(kind='bar', color=['salmon', 'steelblue'])
plt.title('Class Distribution')
plt.xlabel('Class (0=malignant, 1=benign)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

Slight class imbalance (357 benign vs 212 malignant) but not extreme. Stratified split should handle this fine.

In [ ]:
# correlation heatmap — just to see which features might matter most
plt.figure(figsize=(12, 10))
corr = X.corrwith(y)
corr_sorted = corr.abs().sort_values(ascending=False)
print('Top 10 features correlated with target:')
print(corr_sorted.head(10))

Several features have pretty high correlation with the target. This suggests a linear model might actually do decently well, which will be interesting to compare against the MLP.

## 2. Data Preprocessing

- 80/20 stratified split
- StandardScaler (zero mean, unit variance)
- No missing values to handle
- All features are already numeric so no encoding needed

In [ ]:
X_arr = X.values
y_arr = y.values

X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=0.2, random_state=42, stratify=y_arr
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'Split: 80/20 stratified')

## 3. Baseline Model — Logistic Regression (3 marks)

Implementing logistic regression from scratch with:
- Sigmoid activation
- Binary cross-entropy loss
- Gradient descent (batch)
- Tracking loss at every iteration

In [ ]:
class BaselineModel:
    """Logistic Regression with gradient descent."""

    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.loss_history = []
        self.weights = None
        self.bias = None

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iterations):
            linear = X @ self.weights + self.bias
            predictions = self._sigmoid(linear)

            eps = 1e-15
            loss = -np.mean(
                y * np.log(predictions + eps)
                + (1 - y) * np.log(1 - predictions + eps)
            )
            self.loss_history.append(loss)

            error = predictions - y
            dw = (1 / n_samples) * (X.T @ error)
            db = (1 / n_samples) * np.sum(error)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
bl = BaselineModel(learning_rate=0.1, n_iterations=2000)
t0 = time.time()
bl.fit(X_train, y_train)
bl_time = time.time() - t0

print(f'Training time: {bl_time:.4f}s')
print(f'Final loss: {bl.loss_history[-1]:.6f}')
print(f'Loss decreased: {bl.loss_history[0]:.4f} -> {bl.loss_history[-1]:.4f}')

In [ ]:
# quick sanity check — loss should be going down
plt.figure(figsize=(8, 4))
plt.plot(bl.loss_history)
plt.title('Logistic Regression - Training Loss')
plt.xlabel('Iteration')
plt.ylabel('BCE Loss')
plt.grid(True, alpha=0.3)
plt.show()

Loss is dropping nicely. Converges pretty fast actually — most of the decrease happens in the first ~500 iterations. The learning rate of 0.1 seems to work well here, tried 0.01 initially but it was converging too slowly.

## 4. Multi-Layer Perceptron (4 marks)

Architecture: `[30, 64, 32, 1]`
- Input: 30 features
- Hidden layer 1: 64 neurons, ReLU
- Hidden layer 2: 32 neurons, ReLU
- Output: 1 neuron, Sigmoid (binary classification)

Using He initialization for ReLU layers and Xavier for the output layer.

In [ ]:
class MLP:
    def __init__(self, architecture, learning_rate=0.01, n_iterations=1000):
        self.architecture = architecture
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.loss_history = []
        self.params = {}
        self.initialize_parameters()

    def initialize_parameters(self):
        for i in range(1, len(self.architecture)):
            fan_in = self.architecture[i - 1]
            fan_out = self.architecture[i]
            if i < len(self.architecture) - 1:
                scale = np.sqrt(2.0 / fan_in)  # He init for ReLU
            else:
                scale = np.sqrt(1.0 / fan_in)  # Xavier for sigmoid
            self.params[f'W{i}'] = np.random.randn(fan_in, fan_out) * scale
            self.params[f'b{i}'] = np.zeros((1, fan_out))

    @staticmethod
    def _relu(z):
        return np.maximum(0, z)

    @staticmethod
    def _relu_derivative(z):
        return (z > 0).astype(float)

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def forward_propagation(self, X):
        cache = {'A0': X}
        A = X
        n_layers = len(self.architecture) - 1

        for i in range(1, n_layers + 1):
            Z = A @ self.params[f'W{i}'] + self.params[f'b{i}']
            cache[f'Z{i}'] = Z

            if i < n_layers:
                A = self._relu(Z)
            else:
                A = self._sigmoid(Z)

            cache[f'A{i}'] = A

        return A, cache

    def backward_propagation(self, y, cache):
        n_layers = len(self.architecture) - 1
        n_samples = y.shape[0]
        grads = {}

        y = y.reshape(-1, 1)
        A_out = cache[f'A{n_layers}']

        eps = 1e-15
        dA = -(y / (A_out + eps)) + (1 - y) / (1 - A_out + eps)
        dZ = A_out - y

        for i in range(n_layers, 0, -1):
            A_prev = cache[f'A{i - 1}']

            grads[f'dW{i}'] = (1 / n_samples) * (A_prev.T @ dZ)
            grads[f'db{i}'] = (1 / n_samples) * np.sum(dZ, axis=0, keepdims=True)

            if i > 1:
                dA_prev = dZ @ self.params[f'W{i}'].T
                dZ = dA_prev * self._relu_derivative(cache[f'Z{i - 1}'])

        return grads

    def _update_parameters(self, grads):
        n_layers = len(self.architecture) - 1
        for i in range(1, n_layers + 1):
            self.params[f'W{i}'] -= self.lr * grads[f'dW{i}']
            self.params[f'b{i}'] -= self.lr * grads[f'db{i}']

    def fit(self, X, y):
        for _ in range(self.n_iterations):
            A_out, cache = self.forward_propagation(X)

            eps = 1e-15
            y_col = y.reshape(-1, 1)
            loss = -np.mean(
                y_col * np.log(A_out + eps)
                + (1 - y_col) * np.log(1 - A_out + eps)
            )
            self.loss_history.append(loss)

            grads = self.backward_propagation(y, cache)
            self._update_parameters(grads)

        return self

    def predict_proba(self, X):
        A_out, _ = self.forward_propagation(X)
        return A_out.ravel()

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
n_features = X_train.shape[1]
architecture = [n_features, 64, 32, 1]
print(f'Architecture: {architecture}')

mlp = MLP(architecture=architecture, learning_rate=0.01, n_iterations=3000)
t0 = time.time()
mlp.fit(X_train, y_train)
mlp_time = time.time() - t0

print(f'Training time: {mlp_time:.4f}s')
print(f'Final loss: {mlp.loss_history[-1]:.6f}')
print(f'Loss decreased: {mlp.loss_history[0]:.4f} -> {mlp.loss_history[-1]:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(mlp.loss_history)
plt.title('MLP - Training Loss')
plt.xlabel('Iteration')
plt.ylabel('BCE Loss')
plt.grid(True, alpha=0.3)
plt.show()

MLP loss also decreasing smoothly. Took more iterations to settle down compared to logistic regression, which makes sense — more parameters to optimize. Learning rate 0.01 works here; 0.1 was causing instability in the MLP (loss would spike).

## 5. Evaluation & Comparison (2 marks)

In [ ]:
def compute_metrics(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)

    return {'Accuracy': accuracy, 'Precision': precision,
            'Recall': recall, 'F1-Score': f1}

In [ ]:
bl_preds = bl.predict(X_test)
mlp_preds = mlp.predict(X_test)

bl_metrics = compute_metrics(y_test, bl_preds)
mlp_metrics = compute_metrics(y_test, mlp_preds)

results_df = pd.DataFrame({'Logistic Regression': bl_metrics, 'MLP': mlp_metrics})
results_df

Interesting — the baseline logistic regression is actually performing on par with the MLP here. Both have the same recall, and the baseline even edges ahead slightly on precision and F1. More on this in the analysis below.

In [ ]:
# confusion matrices side by side
def confusion_matrix_manual(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return np.array([[tn, fp], [fn, tp]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm_bl = confusion_matrix_manual(y_test, bl_preds)
sns.heatmap(cm_bl, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
axes[0].set_title('Logistic Regression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_mlp = confusion_matrix_manual(y_test, mlp_preds)
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
axes[1].set_title('MLP')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

The confusion matrices tell the same story — both models are making very few mistakes. The key number to watch is false negatives (bottom-left cell) since we're prioritizing recall.

In [ ]:
# loss curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(bl.loss_history, color='steelblue')
axes[0].set_title('Logistic Regression — Training Loss')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(mlp.loss_history, color='coral')
axes[1].set_title('MLP — Training Loss')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Binary Cross-Entropy Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# performance comparison bar chart
labels = list(bl_metrics.keys())
bl_vals = [bl_metrics[k] for k in labels]
mlp_vals = [mlp_metrics[k] for k in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, bl_vals, width, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x + width/2, mlp_vals, width, label='MLP', color='coral')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# training time comparison
print(f'Logistic Regression training time: {bl_time:.4f}s')
print(f'MLP training time: {mlp_time:.4f}s')
print(f'MLP is {mlp_time / bl_time:.1f}x slower')

### Analysis

The MLP achieved an F1 of ~0.965 vs ~0.972 for Logistic Regression — basically the same. On recall (our primary metric) both models scored equally at ~0.958. The MLP's hidden layers can capture non-linear decision boundaries, but here the dataset is already quite linearly separable which is why the baseline holds up so well. Computationally the MLP is roughly 20x slower due to forward+backward passes through multiple layers. Both loss curves decrease monotonically confirming gradient descent works correctly. The main takeaway is that a more complex model doesn't always win — when the underlying data has a mostly linear structure, logistic regression is hard to beat and much cheaper to train. The MLP would likely show a bigger advantage on data with more complex feature interactions.

In [ ]:
analysis = """The MLP achieved an F1 of ~0.965 vs ~0.972 for Logistic Regression — basically the same. On recall (our primary metric) both models scored equally at ~0.958. The MLP's hidden layers can capture non-linear decision boundaries, but here the dataset is already quite linearly separable which is why the baseline holds up so well. Computationally the MLP is roughly 20x slower due to forward+backward passes through multiple layers. Both loss curves decrease monotonically confirming gradient descent works correctly. The main takeaway is that a more complex model doesn't always win — when the underlying data has a mostly linear structure, logistic regression is hard to beat and much cheaper to train. The MLP would likely show a bigger advantage on data with more complex feature interactions."""
print(f'Word count: {len(analysis.split())}')

## `get_assignment_results()`

In [ ]:
def get_assignment_results():
    return {
        'dataset': {
            'name': 'Breast Cancer Wisconsin (Diagnostic)',
            'samples': 569,
            'features': 30,
            'problem_type': 'Binary Classification',
            'primary_metric': 'Recall',
        },
        'baseline': {
            'model': 'Logistic Regression',
            'metrics': bl_metrics,
            'loss_history': bl.loss_history,
            'training_time_s': bl_time,
        },
        'mlp': {
            'model': 'MLP',
            'architecture': architecture,
            'metrics': mlp_metrics,
            'loss_history': mlp.loss_history,
            'training_time_s': mlp_time,
        },
    }

results = get_assignment_results()
print('Results dict keys:', list(results.keys()))